In [4]:
# ============================================================================
# CONFIGURACIÓN INICIAL Y CARGA DE DATOS
# ============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve, confusion_matrix,
                             classification_report)
from sklearn.impute import SimpleImputer
import joblib
import warnings
warnings.filterwarnings('ignore')

# Configurar visualizaciones
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Montar Drive (si no está montado)
from google.colab import drive
drive.mount('/content/drive')

print("✅ Librerías cargadas correctamente")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Librerías cargadas correctamente


In [5]:
# ============================================================================
# LOCALIZAR Y CARGAR EL CSV
# ============================================================================

print("="*60)
print("BUSCANDO TU ARCHIVO CSV")
print("="*60)

# Buscar en Drive
csv_files = []
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for file in files:
        if file.endswith('.csv') and ('twitter' in file.lower() or 'profile' in file.lower()):
            filepath = os.path.join(root, file)
            size = os.path.getsize(filepath)
            csv_files.append((filepath, size))
            print(f"📄 Encontrado: {filepath} ({size:,} bytes)")

if csv_files:
    # Usar el archivo más grande (dataset completo)
    filepath = csv_files[0][0]
    df = pd.read_csv(filepath)
    print(f"\n✅ Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    print(f"\n📊 Columnas disponibles:")
    print(df.columns.tolist())
else:
    # Si no encuentra, subir manualmente
    print("❌ No se encontró el archivo. Por favor, súbelo manualmente:")
    from google.colab import files
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    df = pd.read_csv(filename)
    print(f"✅ Dataset cargado: {df.shape}")

BUSCANDO TU ARCHIVO CSV
📄 Encontrado: /content/drive/MyDrive/twitter_project/data/processed/twitter_profiles_cleaned.csv (1,486,016 bytes)

✅ Dataset cargado: 7689 filas, 16 columnas

📊 Columnas disponibles:
['name', 'screen_name', 'followers_count', 'friends_count', 'post_count', 'lang', 'location', 'default_profile_image', 'profile_use_background_image', 'verified', 'description', 'created_at', 'label', 'has_location', 'location_clean', 'is_real_location']


In [6]:
# ============================================================================
# PREPARACIÓN DE DATOS
# ============================================================================

print("\n" + "="*60)
print("PREPARACIÓN DE DATOS PARA MODELADO")
print("="*60)

# Verificar la variable objetivo
print(f"\n🔍 Variable objetivo 'label':")
print(f"   Valores únicos: {df['label'].unique()}")
print(f"   Distribución:")
print(df['label'].value_counts())

# Seleccionar características para modelado
feature_columns = ['followers_count', 'friends_count', 'post_count', 'has_location']

# Agregar is_real_location si existe
if 'is_real_location' in df.columns:
    feature_columns.append('is_real_location')
    print(f"\n✅ Usando columna 'is_real_location'")
else:
    print(f"\n⚠️ 'is_real_location' no encontrada")

# Verificar y manejar valores nulos
print("\n🔍 Verificando valores nulos:")
for col in feature_columns:
    null_count = df[col].isnull().sum()
    if null_count > 0:
        print(f"   ⚠️ {col}: {null_count} nulos → rellenando con 0")
        df[col] = df[col].fillna(0)
    else:
        print(f"   ✅ {col}: sin nulos")

# Crear X e y
X = df[feature_columns].values
y = df['label'].values

print(f"\n📊 Dimensiones:")
print(f"   X: {X.shape}")
print(f"   y: {y.shape}")

# Verificar balance de clases
unique, counts = np.unique(y, return_counts=True)
print(f"\n📊 Distribución de clases:")
print(f"   Reales (0): {counts[0]} ({counts[0]/len(y)*100:.1f}%)")
print(f"   Bots (1): {counts[1]} ({counts[1]/len(y)*100:.1f}%)")


PREPARACIÓN DE DATOS PARA MODELADO

🔍 Variable objetivo 'label':
   Valores únicos: [0 1]
   Distribución:
label
0    5040
1    2649
Name: count, dtype: int64

✅ Usando columna 'is_real_location'

🔍 Verificando valores nulos:
   ✅ followers_count: sin nulos
   ✅ friends_count: sin nulos
   ✅ post_count: sin nulos
   ✅ has_location: sin nulos
   ⚠️ is_real_location: 1765 nulos → rellenando con 0

📊 Dimensiones:
   X: (7689, 5)
   y: (7689,)

📊 Distribución de clases:
   Reales (0): 5040 (65.5%)
   Bots (1): 2649 (34.5%)


In [7]:
# ============================================================================
# DIVISIÓN ENTRENAMIENTO/TEST Y ESCALADO
# ============================================================================

print("\n" + "="*60)
print("DIVIDIENDO DATOS")
print("="*60)

# División estratificada para mantener proporción de clases
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"✅ Entrenamiento: {X_train.shape[0]} muestras")
print(f"✅ Prueba: {X_test.shape[0]} muestras")

# Escalar características (importante para SVM, KNN y Regresión Logística)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ Datos escalados (media=0, std=1)")


DIVIDIENDO DATOS
✅ Entrenamiento: 6151 muestras
✅ Prueba: 1538 muestras

✅ Datos escalados (media=0, std=1)


In [8]:
# ============================================================================
# ENTRENAMIENTO DE MODELOS
# ============================================================================

print("\n" + "="*60)
print("ENTRENANDO MODELOS DE CLASIFICACIÓN")
print("="*60)

# Definición de modelos con justificación técnica
models = {
    'Regresión Logística': LogisticRegression(max_iter=1000, random_state=42),
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='rbf', random_state=42, probability=True),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

trained_models = {}
training_results = []

for name, model in models.items():
    print(f"\n📈 Entrenando {name}...")
    model.fit(X_train_scaled, y_train)
    trained_models[name] = model

    # Evaluación rápida
    y_pred_train = model.predict(X_train_scaled)
    f1_train = f1_score(y_train, y_pred_train)
    print(f"   F1-Score (train): {f1_train:.4f}")

print(f"\n✅ {len(trained_models)} modelos entrenados")


ENTRENANDO MODELOS DE CLASIFICACIÓN

📈 Entrenando Regresión Logística...
   F1-Score (train): 0.8178

📈 Entrenando Árbol de Decisión...
   F1-Score (train): 1.0000

📈 Entrenando Random Forest...
   F1-Score (train): 0.9998

📈 Entrenando Gradient Boosting...
   F1-Score (train): 0.9922

📈 Entrenando SVM...
   F1-Score (train): 0.8771

📈 Entrenando KNN...
   F1-Score (train): 0.9372

✅ 6 modelos entrenados
